#### Initialize

In [6]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
MAP_PATH = PARENT / "server/map/grid"
MAP_PATH.mkdir(parents=True, exist_ok=True)
RANKED_BUCKET_PATH = PARENT / "server/out/places_ranked"
DF_RANKED = pd.read_csv(RANKED_BUCKET_PATH / "places_scored_level_1.csv")
DF_RANKED = DF_RANKED.drop(columns=['wilson_0', 'normal_0', 'wilson_2', 'normal_2', 'h3_res9'])


#### Representation

In [7]:
df_ranked = DF_RANKED.copy()
# df_ranked = df_ranked[df_ranked['normal_1']>=0.75]

counts = (
    df_ranked.assign(**{'cuisineType': df_ranked['cuisineType'].fillna('(missing)')})
      .groupby('cuisineType', dropna=False)
      .size()
      .reset_index(name='row_count')
      .sort_values('row_count', ascending=False)
)

print(f"Using column: {'cuisineType'}")
counts.sort_values('row_count', ascending=False)

Using column: cuisineType


,cuisineType,row_count
38,Unspecified,1607
17,Fast Food,990
28,Middle Eastern,748
33,South Asian,743
12,Chinese,741
23,Japanese,740
22,Italian,706
30,Pizza,677
26,Latin American,670
34,Southeast Asian,565


#### Tier Diversified

In [8]:
import numpy as np

def assign_tier(normal_score: pd.Series) -> pd.Series:
    return pd.cut(
        normal_score,
        bins=[-np.inf, 0.5, 0.75, 0.9, 0.95, np.inf],
        labels=[0, 1, 2, 3, 4],
        right=False,
    ).astype(int)

# Base tier from normalized Wilson score
df_ranked['tier'] = assign_tier(df_ranked['normal_1'])

# Diversified tier (tier_d): start from base tier, then clip over-represented cuisines
df_ranked['tier_d'] = df_ranked['tier']
df_ranked['_cuisine_key'] = df_ranked['cuisineType'].fillna('(missing)')

score_col = 'normal_1'
cap_multiplier = 1.2
min_cuisine_size = 50

# Global cuisine prevalence
global_counts = df_ranked['_cuisine_key'].value_counts()
global_share = (global_counts / len(df_ranked)).to_dict()
eligible_cuisines = set(global_counts[global_counts > min_cuisine_size].index)

# Enforce cap per tier from top to bottom.
# If a cuisine exceeds cap in a tier, demote lowest-scored excess rows to next lower tier.
for t in sorted(df_ranked['tier_d'].unique(), reverse=True):
    if t == 0:
        continue

    tier_mask = df_ranked['tier_d'] == t
    tier_size = int(tier_mask.sum())
    if tier_size == 0:
        continue

    tier_counts = df_ranked.loc[tier_mask, '_cuisine_key'].value_counts()

    for cuisine, cnt in tier_counts.items():
        if cuisine not in eligible_cuisines:
            continue

        cap = int(np.floor(cap_multiplier * global_share[cuisine] * tier_size))
        cap = max(cap, 1)
        excess = int(cnt - cap)

        if excess <= 0:
            continue

        drop_idx = (
            df_ranked.loc[tier_mask & (df_ranked['_cuisine_key'] == cuisine)]
            .sort_values(score_col, ascending=True)
            .head(excess)
            .index
        )
        df_ranked.loc[drop_idx, 'tier_d'] = t - 1

# Cleanup helper column
df_ranked.drop(columns=['_cuisine_key'], inplace=True)

In [9]:
# Quick check
display(df_ranked[['tier', 'tier_d']].value_counts().rename('rows').reset_index().sort_values(['tier', 'tier_d']))
# df_ranked[['tier', 'tier_d']].head()

,tier,tier_d,rows
0,0,0,6545
5,1,0,266
1,1,1,3008
6,2,1,159
2,2,2,1804
7,3,2,98
4,3,3,556
8,4,3,85
3,4,4,571


#### Tier NoChain

In [10]:
# tier_independent: same as tier_d, but remove chain restaurants from tiers 1-4
df_ranked['tier_independent'] = df_ranked['tier']
df_ranked['_cuisine_key'] = df_ranked['cuisineType'].fillna('(missing)')

# Apply the same representation cap logic as tier_d
for t in sorted(df_ranked['tier_independent'].unique(), reverse=True):
    if t == 0:
        continue

    tier_mask = df_ranked['tier_independent'] == t
    tier_size = int(tier_mask.sum())
    if tier_size == 0:
        continue

    tier_counts = df_ranked.loc[tier_mask, '_cuisine_key'].value_counts()

    for cuisine, cnt in tier_counts.items():
        if cuisine not in eligible_cuisines:
            continue

        cap = int(np.floor(cap_multiplier * global_share[cuisine] * tier_size))
        cap = max(cap, 1)
        excess = int(cnt - cap)

        if excess <= 0:
            continue

        drop_idx = (
            df_ranked.loc[tier_mask & (df_ranked['_cuisine_key'] == cuisine)]
            .sort_values(score_col, ascending=True)
            .head(excess)
            .index
        )
        df_ranked.loc[drop_idx, 'tier_independent'] = t - 1

# Demote all chain restaurants from tiers 1-4 to tier 0
chain_mask = (df_ranked['is_chain'] == True) & (df_ranked['tier_independent'] > 0)
df_ranked.loc[chain_mask, 'tier_independent'] = 0

# Cleanup helper column
df_ranked.drop(columns=['_cuisine_key'], inplace=True)

In [11]:
# Check the result
display(df_ranked[['tier', 'tier_d', 'tier_independent']].value_counts().rename('rows').reset_index().sort_values(['tier', 'tier_d', 'tier_independent']))
print(f"\nChain restaurants demoted to tier 0: {chain_mask.sum()}")
df_ranked[['tier', 'tier_d', 'tier_independent', 'is_chain']].head()

,tier,tier_d,tier_independent,rows
0,0,0,0,6545
5,1,0,0,266
6,1,1,0,196
1,1,1,1,2812
14,2,1,0,8
7,2,1,1,151
9,2,2,0,83
2,2,2,2,1721
13,3,2,0,8
8,3,2,2,90



Chain restaurants demoted to tier 0: 342


,tier,tier_d,tier_independent,is_chain
0,4,4,4,False
1,4,4,4,False
2,4,4,4,False
3,4,4,4,False
4,4,4,4,False


#### Export

In [12]:
df_ranked.to_csv(RANKED_BUCKET_PATH / "places_ranked_level_2.csv", index=False)